# Extração de Features Radiômicas — trainval_set e test_set

Extrai features radiômicas com PyRadiomics para as imagens BW e Doppler dos splits `trainval` e `test`, gerando dois CSVs finais:
- `radiomics_bw.csv` — features de imagens BW
- `radiomics_doppler.csv` — features de imagens Doppler

Cada CSV contém colunas `image_id`, `label`, `target`, `split` e todas as features radiômicas.

## Env Configuration

Instala dependências ausentes, importa bibliotecas e define constantes de configuração da extração.

In [6]:
import importlib.util
import subprocess
import sys

required_packages = {
    "radiomics": "pyradiomics",
    "SimpleITK": "SimpleITK",
    "tqdm": "tqdm",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

In [ ]:
import pandas as pd

OUTPUT_DIRNAME  = "datasets"
BIN_WIDTH       = 25
NORMALIZE       = True
NORMALIZE_SCALE = 100
ENABLE_WAVELET  = True

## Functions

Importa funções de `modules/radiomics_utils.py`: localização do projeto, montagem dos catálogos de imagens, conversão para SimpleITK e execução da extração com PyRadiomics.

In [ ]:
import sys
from pathlib import Path

# Adiciona modules/ ao path para importar módulos do projeto
sys.path.insert(0, str((Path("..") / "modules").resolve()))

from radiomics_utils import (
    find_project_root,
    load_trainval_catalog,
    load_test_catalog,
    build_extractor,
    extract_features_for_catalog,
)

## Process

Monta os catálogos de trainval e test para cada modalidade, executa a extração e salva os CSVs finais em `data/images/datasets/`.

In [ ]:
project_root = find_project_root(Path.cwd())
output_dir = project_root / "data" / "images" / OUTPUT_DIRNAME
output_dir.mkdir(parents=True, exist_ok=True)

extractor = build_extractor(
    bin_width=BIN_WIDTH,
    normalize=NORMALIZE,
    normalize_scale=NORMALIZE_SCALE,
    enable_wavelet=ENABLE_WAVELET,
)
print(f"Project root: {project_root}")
print(f"Output dir:   {output_dir}")

In [10]:
# BW
catalog_bw = pd.concat([
    load_trainval_catalog(project_root, "bw"),
    load_test_catalog(project_root, "bw"),
], ignore_index=True)

print(f"BW catalog: {len(catalog_bw)} imagens")
catalog_bw[["split", "label"]].value_counts().sort_index()

BW catalog: 198 imagens


split     label    
test      BENIGN        13
          MALIGNANT      7
trainval  BENIGN       118
          MALIGNANT     60
dtype: int64

In [11]:
radiomics_bw = extract_features_for_catalog(catalog_bw, extractor)

output_bw = output_dir / "radiomics_bw.csv"
radiomics_bw.to_csv(output_bw, index=False)
print(f"Salvo em: {output_bw}")
print(f"Shape: {radiomics_bw.shape}")
radiomics_bw[["split", "label", "target"]].value_counts().sort_index()

Extraindo: 100%|██████████| 198/198 [04:58<00:00,  1.51s/it]


Salvo em: C:\Users\LUCAS E GABRIEL\Documents\TCC-PROJECT\data\processed\radiomics\radiomics_bw.csv
Shape: (198, 469)


split     label      target
test      BENIGN     0          13
          MALIGNANT  1           7
trainval  BENIGN     0         118
          MALIGNANT  1          60
dtype: int64

In [12]:
# Doppler
catalog_doppler = pd.concat([
    load_trainval_catalog(project_root, "doppler"),
    load_test_catalog(project_root, "doppler"),
], ignore_index=True)

print(f"Doppler catalog: {len(catalog_doppler)} imagens")
catalog_doppler[["split", "label"]].value_counts().sort_index()

Doppler catalog: 198 imagens


split     label    
test      BENIGN        13
          MALIGNANT      7
trainval  BENIGN       118
          MALIGNANT     60
dtype: int64

In [13]:
radiomics_doppler = extract_features_for_catalog(catalog_doppler, extractor)

output_doppler = output_dir / "radiomics_doppler.csv"
radiomics_doppler.to_csv(output_doppler, index=False)
print(f"Salvo em: {output_doppler}")
print(f"Shape: {radiomics_doppler.shape}")
radiomics_doppler[["split", "label", "target"]].value_counts().sort_index()

Extraindo: 100%|██████████| 198/198 [05:02<00:00,  1.53s/it]


Salvo em: C:\Users\LUCAS E GABRIEL\Documents\TCC-PROJECT\data\processed\radiomics\radiomics_doppler.csv
Shape: (198, 469)


split     label      target
test      BENIGN     0          13
          MALIGNANT  1           7
trainval  BENIGN     0         118
          MALIGNANT  1          60
dtype: int64